# Bus/Train Ticket Booking Data Analysis

This notebook uses a reproducible **synthetic** Indian booking dataset. It is not real operator or passenger data. Recognized revenue is Ticket_Price multiplied by Number_of_Tickets for Completed and Confirmed bookings only; refund accounting is outside scope.

## 1. Imports and project paths
The notebook calls the same tested functions used by the command-line program.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.generate_dataset import generate_booking_data
from src.data_cleaning import clean_booking_data
from src.analysis import build_analysis
from src.visualizations import create_visualizations

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'booking_data.csv'
print('Project root:', PROJECT_ROOT)

Project root: C:\Users\ajay.gautam\Desktop\BCA Project\Bus_Train_Ticket_Booking_Analysis


## 2. Load or generate source data
The fixed seed makes the generated dataset reproducible. Raw rows include intentional missing values and exact duplicates for cleaning demonstrations.

In [2]:
if not RAW_PATH.exists():
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    generate_booking_data().to_csv(RAW_PATH, index=False, date_format='%Y-%m-%d')
raw_data = pd.read_csv(RAW_PATH)
print('Raw shape:', raw_data.shape)
display(raw_data.head())
raw_data.info()

Raw shape: (1208, 13)


,Booking_ID,Passenger_Name,Age,Gender,Transport_Mode,Source,Destination,Booking_Date,Travel_Date,Ticket_Price,Number_of_Tickets,Booking_Status,Payment_Mode
0,BK000001,Kabir Das,10.0,Male,Bus,Mumbai,Varanasi,2026-03-14,2026-05-13,1072.11,3,Confirmed,Net Banking
1,BK000002,Rohan Das,46.0,Male,Train,Jaipur,Kanpur,2026-04-19,2026-04-24,889.62,4,Pending,Card
2,BK000003,Sana Verma,77.0,Male,Train,Kanpur,Varanasi,2026-03-09,2026-05-22,592.97,2,Completed,UPI
3,BK000004,Kavya Mishra,16.0,Female,Bus,Kolkata,Varanasi,2025-08-09,2025-10-13,1849.53,3,Confirmed,Card
4,BK000005,Ananya Sharma,22.0,Male,Bus,Patna,Lucknow,2026-05-24,2026-07-26,1149.03,2,Cancelled,UPI


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1208 entries, 0 to 1207
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Booking_ID         1208 non-null   object 
 1   Passenger_Name     1203 non-null   object 
 2   Age                1196 non-null   float64
 3   Gender             1200 non-null   object 
 4   Transport_Mode     1208 non-null   object 
 5   Source             1208 non-null   object 
 6   Destination        1208 non-null   object 
 7   Booking_Date       1208 non-null   object 
 8   Travel_Date        1208 non-null   object 
 9   Ticket_Price       1208 non-null   float64
 10  Number_of_Tickets  1208 non-null   int64  
 11  Booking_Status     1208 non-null   object 
 12  Payment_Mode       1198 non-null   object 
dtypes: float64(2), int64(1), object(10)
memory usage: 122.8+ KB


## 3. Inspect missing values and exact duplicates
Only fully identical raw rows are duplicates. Similar-looking journeys with different booking IDs are retained.

In [3]:
display(raw_data.isna().sum().to_frame('Missing_Cells'))
print('Exact duplicate rows:', int(raw_data.duplicated().sum()))

,Missing_Cells
Booking_ID,0
Passenger_Name,5
Age,12
Gender,8
Transport_Mode,0
Source,0
Destination,0
Booking_Date,0
Travel_Date,0
Ticket_Price,0


Exact duplicate rows: 8


## 4. Clean and engineer features
The cleaner converts types, removes exact duplicates and invalid transactions, imputes selected fields, and derives route, month, and revenue values.

In [4]:
cleaned_data, cleaning_summary = clean_booking_data(raw_data)
display(pd.Series(cleaning_summary, name='Count').to_frame())
display(cleaned_data.head())
print('Missing values after cleaning:', int(cleaned_data.isna().sum().sum()))

,Count
input_rows,1208
exact_duplicates_removed,8
invalid_rows_removed,0
missing_cells_before_cleaning,35
cleaned_rows,1200
rows_removed,8


,Booking_ID,Passenger_Name,Age,Gender,Transport_Mode,Source,Destination,Booking_Date,Travel_Date,Ticket_Price,Number_of_Tickets,Booking_Status,Payment_Mode,Revenue,Revenue_Eligible,Recognized_Revenue,Booking_Month,Travel_Route
0,BK000001,Kabir Das,10,Male,Bus,Mumbai,Varanasi,2026-03-14,2026-05-13,1072.11,3,Confirmed,Net Banking,3216.33,True,3216.33,2026-03,Mumbai to Varanasi
1,BK000002,Rohan Das,46,Male,Train,Jaipur,Kanpur,2026-04-19,2026-04-24,889.62,4,Pending,Card,3558.48,False,0.00,2026-04,Jaipur to Kanpur
2,BK000003,Sana Verma,77,Male,Train,Kanpur,Varanasi,2026-03-09,2026-05-22,592.97,2,Completed,UPI,1185.94,True,1185.94,2026-03,Kanpur to Varanasi
3,BK000004,Kavya Mishra,16,Female,Bus,Kolkata,Varanasi,2025-08-09,2025-10-13,1849.53,3,Confirmed,Card,5548.59,True,5548.59,2025-08,Kolkata to Varanasi
4,BK000005,Ananya Sharma,22,Male,Bus,Patna,Lucknow,2026-05-24,2026-07-26,1149.03,2,Cancelled,UPI,2298.06,False,0.00,2026-05,Patna to Lucknow


Missing values after cleaning: 0


## 5. Headline booking and revenue measures
Booking count means unique transactions. Ticket count sums the tickets in each transaction. Cancelled and pending rows contribute zero recognized revenue.

In [5]:
results = build_analysis(cleaned_data)
metrics = results['metrics']
display(pd.Series(metrics, name='Value').to_frame())
print('Age percentiles (25th, 50th, 75th):', results['numpy_age_percentiles'])

,Value
dataset_records,1200
total_bookings,1200
completed_bookings,832
confirmed_bookings,142
cancelled_bookings,152
pending_bookings,74
eligible_bookings,974
ticket_count,3531
eligible_ticket_count,2854
recognized_revenue,3880660.86


Age percentiles (25th, 50th, 75th): [24.0, 44.0, 62.0]


## 6. Destination and route patterns
These summaries describe this synthetic sample only and are not claims about actual travel demand.

In [6]:
display(results['destination_summary'].head(12))
display(results['route_summary'].head(10))

,Destination,Booking_Count,Ticket_Count,Recognized_Revenue
11,Varanasi,117,335,401212.88
9,Prayagraj,115,317,373939.57
10,Ranchi,110,350,394795.42
3,Jaipur,104,287,340808.06
8,Patna,103,307,362131.03
5,Kolkata,102,291,252456.61
0,Agra,101,310,346414.67
2,Delhi,96,291,272661.36
7,Mumbai,92,276,287188.68
4,Kanpur,90,257,304382.12


,Travel_Route,Booking_Count,Ticket_Count,Recognized_Revenue
93,Patna to Kolkata,20,66,62139.58
24,Delhi to Jaipur,17,45,45570.70
119,Ranchi to Prayagraj,17,45,50812.69
76,Lucknow to Varanasi,16,37,33185.56
6,Agra to Mumbai,15,43,46896.14
63,Kolkata to Prayagraj,15,46,58972.58
65,Kolkata to Varanasi,15,50,42732.97
32,Delhi to Varanasi,14,41,49678.12
34,Jaipur to Bhopal,14,35,48590.67
43,Jaipur to Varanasi,14,48,61038.66


## 7. Passenger age and ticket prices
Age groups, descriptive statistics, and NumPy percentiles summarize the generated records.

In [7]:
display(results['age_summary'])
display(results['price_summary'])
print('NumPy age quartiles:', np.percentile(cleaned_data['Age'], [25, 50, 75]))

,Age_Group,Passenger_Records
0,0-17,215
1,18-25,117
2,26-40,223
3,41-60,330
4,61+,315


,Statistic,Ticket_Price
0,count,1200.000000
1,mean,1363.168133
2,std,666.198161
3,min,292.910000
4,25%,823.450000
5,50%,1266.780000
6,75%,1782.242500
7,max,3416.540000


NumPy age quartiles: [24. 44. 62.]


## 8. Bus versus train and booking status
Average ticket price is per ticket; recognized revenue includes ticket quantity and eligible status.

In [8]:
display(results['mode_summary'])
display(results['status_summary'])

,Transport_Mode,Booking_Count,Ticket_Count,Average_Ticket_Price,Recognized_Revenue
0,Bus,672,1974,1111.613229,1739834.34
1,Train,528,1557,1683.328920,2140826.52


,Booking_Status,Booking_Count
0,Completed,832
1,Cancelled,152
2,Confirmed,142
3,Pending,74


## 9. Monthly bookings and revenue
Months are based on Booking_Date. The dataset is for descriptive demonstration and does not forecast.

In [9]:
monthly = results['monthly_summary']
display(monthly)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(monthly['Booking_Month'], monthly['Booking_Count'], marker='o')
axes[0].set(title='Monthly Bookings', ylabel='Unique bookings')
axes[1].plot(monthly['Booking_Month'], monthly['Recognized_Revenue'], marker='o', color='teal')
axes[1].set(title='Monthly Recognized Revenue', xlabel='Booking month', ylabel='Revenue (INR)')
plt.xticks(rotation=60, ha='right'); plt.tight_layout(); plt.show()

,Booking_Month,Booking_Count,Ticket_Count,Recognized_Revenue
0,2025-01,68,208,215535.63
1,2025-02,61,195,231167.32
2,2025-03,61,204,200752.18
3,2025-04,45,139,120064.52
4,2025-05,66,198,227314.35
5,2025-06,69,204,213115.68
6,2025-07,57,162,210380.26
7,2025-08,60,173,184244.91
8,2025-09,58,151,138878.20
9,2025-10,54,175,206181.48


C:\Users\ajay.gautam\AppData\Local\Temp\ipykernel_47988\1164882010.py:8: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.xticks(rotation=60, ha='right'); plt.tight_layout(); plt.show()


## 10. Generate charts and export cleaned data
Save the complete set of ten Matplotlib charts and the cleaned CSV.

In [10]:
processed_path = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_booking_data.csv'
processed_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_data.to_csv(processed_path, index=False, date_format='%Y-%m-%d')
chart_paths = create_visualizations(cleaned_data, results, PROJECT_ROOT / 'figures')
print('Cleaned CSV:', processed_path)
print('Charts created:', len(chart_paths))
for chart_path in chart_paths:
    print(chart_path.relative_to(PROJECT_ROOT))

Cleaned CSV: C:\Users\ajay.gautam\Desktop\BCA Project\Bus_Train_Ticket_Booking_Analysis\data\processed\cleaned_booking_data.csv
Charts created: 10
figures\bookings_by_destination.png
figures\transport_mode_distribution.png
figures\passenger_age_distribution.png
figures\average_price_by_mode.png
figures\monthly_booking_trends.png
figures\monthly_revenue_trends.png
figures\top_10_routes.png
figures\revenue_by_destination.png
figures\booking_status_distribution.png
figures\ticket_price_distribution.png


## 11. Interpretation and limitations
All apparent leaders, prices, age patterns, and seasonal changes come from assigned synthetic distributions. They are not evidence about real Indian passenger behavior or operator performance.